#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
# @title ⚙️ 1. Setup Environment & Download Models
import os
import urllib.request
from IPython.display import clear_output

print("📦 Installing dependencies and cloning ComfyUI...")
!pip install -q aiohttp websocket-client
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
    %cd ComfyUI
    !pip install -q -r requirements.txt
    %cd /content
else:
    %cd ComfyUI
    !git pull
    %cd /content

os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)
os.makedirs('/content/ComfyUI/models/loras', exist_ok=True)

MODEL_URL = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-turbo-v1.1.safetensors"
TEXT_ENCODER_URL = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
VAE_URL = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"

def download_file(url, dest_path, desc, min_size_mb=100):
    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / (1024*1024)
        if size_mb >= min_size_mb:
            print(f"✅ {desc} already exists. Skipping.")
            return
    print(f"⬇️ Downloading {desc} ...")
    urllib.request.urlretrieve(url, dest_path)
    print(f"✅ {desc} downloaded.")

download_file(MODEL_URL, '/content/ComfyUI/models/diffusion_models/anima-turbo-v1.1.safetensors', 'Anima-Turbo v1.1', 3000)
download_file(TEXT_ENCODER_URL, '/content/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors', 'Qwen text encoder', 500)
download_file(VAE_URL, '/content/ComfyUI/models/vae/qwen_image_vae.safetensors', 'Qwen image VAE', 100)

clear_output()
print("✅ Setup Complete! Move to the next cell.")

In [ ]:
# @title 🚀 2. Start Generation Engine
import subprocess
import time
import requests

print("⏳ Starting ComfyUI server in the background...")
subprocess.Popen(["python", "main.py", "--port", "8188", "--listen", "0.0.0.0"],
                 cwd="/content/ComfyUI",
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats")
        if r.status_code == 200:
            print("✅ Engine is online and ready!")
            break
    except:
        pass
    time.sleep(1)
else:
    print("❌ Failed to start the engine. Try restarting the runtime.")

In [ ]:
# @title 📤 3. Upload LoRA Files (Optional)
# Download lora: https://civitai.com/user/circlestone_labs/models
import os
import shutil
from google.colab import files

print("Select your LoRA (.safetensors) files to upload:")
uploaded = files.upload()

for filename in uploaded.keys():
    dest_path = f"/content/ComfyUI/models/loras/{filename}"
    shutil.move(filename, dest_path)
    print(f"✅ Uploaded: {filename}")

In [ ]:
# @title 🎨 4. Generate Image
# @markdown Fill in your prompts and settings below, then run this cell.

prompt = "A girl wearing a black and red kimono is standing outdoors. She's holding a rectangular sign out in front of her that reads \"COINNOIN\". She's looking at the viewer with a smile. The background features some trees and blue sky with clouds." # @param {type:"string"}

positive_prompt = "masterpiece, best quality, score_7, safe, " + prompt
negative_prompt = "worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, chromatic aberration"

# @markdown ---
# @markdown **Dimensions & Quality**
width = 768 # @param {type:"slider", min:512, max:1536, step:64}
height = 768 # @param {type:"slider", min:512, max:1536, step:64}
steps = 10 # @param {type:"slider", min:8, max:12, step:1}
sampler = "er_sde" # @param ["er_sde", "euler_a", "dpmpp_2m_sde_gpu", "euler"]
# @markdown - er_sde: Clean, simple, good default.
# @markdown - euler_a: Soft, thin lines, slight 3D look, can take more time.
# @markdown - dpmpp_2m_sde_gpu: Creative, more variety, can be wild.
# @markdown - euler: Simple and creative, good for Turbo.
seed = 0

# @markdown ---
# @markdown **LoRA Settings (Optional)**
# @markdown *Type the exact filename of the LoRA you uploaded in Cell 3 (e.g., `my_style.safetensors`). Leave as `None` to ignore.*
lora_filename = "None" # @param {type:"string"}
lora_strength = 1.0 # @param {type:"slider", min:0.1, max:2.0, step:0.1}

# @markdown ---
# @markdown **Output**
auto_download = True # @param {type:"boolean"}

import json, time, requests, urllib, os, io
from PIL import Image
from IPython.display import display, clear_output
from google.colab import files

COMFY_URL = "http://127.0.0.1:8188"

workflow = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "anima-turbo-v1.1.safetensors", "weight_dtype": "default"}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen_3_06b_base.safetensors", "type": "stable_diffusion", "device": "default"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
    "4": {"class_type": "CLIPTextEncode", "inputs": {"text": positive_prompt, "clip": ["2", 0]}},
    "5": {"class_type": "CLIPTextEncode", "inputs": {"text": negative_prompt, "clip": ["2", 0]}},
    "6": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
    "7": {"class_type": "KSampler", "inputs": {"seed": seed, "steps": steps, "cfg": 1.0, "sampler_name": sampler, "scheduler": "simple", "denoise": 1.0, "model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0]}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
    "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "Anima", "images": ["8", 0]}}
}

if lora_filename.lower() != "none" and lora_filename.strip() != "":
    lora_path = f"/content/ComfyUI/models/loras/{lora_filename}"
    if os.path.exists(lora_path):
        workflow["10"] = {
            "class_type": "LoraLoader",
            "inputs": {"lora_name": lora_filename, "strength_model": lora_strength, "strength_clip": lora_strength, "model": ["1", 0], "clip": ["2", 0]}
        }
        workflow["7"]["inputs"]["model"] = ["10", 0]
        workflow["4"]["inputs"]["clip"] = ["10", 1]
        workflow["5"]["inputs"]["clip"] = ["10", 1]
    else:
        print(f"⚠️ LoRA '{lora_filename}' not found. Generating without it.")

def queue_prompt(prompt_workflow):
    r = requests.post(f"{COMFY_URL}/prompt", data=json.dumps({"prompt": prompt_workflow}).encode('utf-8'))
    return r.json().get('prompt_id')

def get_image(filename, subfolder, folder_type):
    r = requests.get(f"{COMFY_URL}/view?{urllib.parse.urlencode({'filename': filename, 'subfolder': subfolder, 'type': folder_type})}")
    return r.content

print("⏳ Generating image... Please wait.")
prompt_id = queue_prompt(workflow)

img_bytes = None
while True:
    time.sleep(1)
    history = requests.get(f"{COMFY_URL}/history/{prompt_id}").json()
    if prompt_id in history:
        outputs = history[prompt_id].get('outputs', {})
        for node_id in outputs:
            if 'images' in outputs[node_id]:
                img_info = outputs[node_id]['images'][0]
                img_bytes = get_image(img_info['filename'], img_info['subfolder'], img_info['type'])
        break

if img_bytes:
    clear_output()
    img = Image.open(io.BytesIO(img_bytes))
    display(img)

    save_path = "/content/generated_Anima-Turbo-v1.1-CoinNoin.png"
    img.save(save_path)
    if auto_download:
        print("⬇️ Downloading image...")
        files.download(save_path)
else:
    print("❌ Failed to generate image.")